
# Phase 3: Feature Engineering

## Step 1: Load Processed Data

Load the cleaned and sales-focused datasets saved during Phase 2.


In [1]:
import pandas as pd
from pathlib import Path

# Path to the processed data folder
DATA_DIR = Path("../data/processed")

# Load both datasets
df_clean = pd.read_csv(
    DATA_DIR / "online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)


df_sales = pd.read_csv(
    DATA_DIR / "online_retail_sales.csv",
    parse_dates=["InvoiceDate"]
)

print("Clean dataset shape:", df_clean.shape)
print("Sales dataset shape:", df_sales.shape)

display(df_sales.head())

Clean dataset shape: (1033036, 8)
Sales dataset shape: (1007912, 14)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancellationInvoice,IsBadDebtAdjustment,IsZeroPrice,IsNegativeQuantity,LineValue,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,False,False,False,83.4,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,81.0,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,False,False,False,81.0,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,False,False,False,100.8,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,False,False,False,30.0,30.0


In [2]:
df_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007912 entries, 0 to 1007911
Data columns (total 14 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Invoice                1007912 non-null  int64         
 1   StockCode              1007912 non-null  str           
 2   Description            1007912 non-null  str           
 3   Quantity               1007912 non-null  int64         
 4   InvoiceDate            1007912 non-null  datetime64[us]
 5   Price                  1007912 non-null  float64       
 6   Customer ID            779425 non-null   float64       
 7   Country                1007912 non-null  str           
 8   IsCancellationInvoice  1007912 non-null  bool          
 9   IsBadDebtAdjustment    1007912 non-null  bool          
 10  IsZeroPrice            1007912 non-null  bool          
 11  IsNegativeQuantity     1007912 non-null  bool          
 12  LineValue              1007912 non-null

In [4]:
print("InvoiceDate dtype:", df_sales["InvoiceDate"].dtype)
print("Quantity dtype:", df_sales["Quantity"].dtype)
print("Price dtype:", df_sales["Price"].dtype)
print("Customer ID dtype:", df_sales["Customer ID"].dtype)

InvoiceDate dtype: datetime64[us]
Quantity dtype: int64
Price dtype: float64
Customer ID dtype: float64



## Step 2: Date Feature Engineering

Extract year, month, year-month, day, and weekday from InvoiceDate to support time-based sales analysis.


In [6]:
df_sales["Year"] = df_sales["InvoiceDate"].dt.year
df_sales["Month"] = df_sales["InvoiceDate"].dt.month

df_sales["YearMonth"] = (
    df_sales["InvoiceDate"].dt.to_period("M")
)

df_sales["Day"] = df_sales["InvoiceDate"].dt.day

df_sales["DayOfWeek"] = (
    df_sales["InvoiceDate"].dt.day_name()
)

# Display the new features
display(
    df_sales[
        [
            "InvoiceDate",
            "Year",
            "Month",
            "YearMonth",
            "Day",
            "DayOfWeek"
        ]
    ].tail(10)
)

,InvoiceDate,Year,Month,YearMonth,Day,DayOfWeek
1007902,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007903,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007904,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007905,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007906,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007907,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007908,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007909,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007910,2011-12-09 12:50:00,2011,12,2011-12,9,Friday
1007911,2011-12-09 12:50:00,2011,12,2011-12,9,Friday


In [7]:
print("Years:", sorted(df_sales["Year"].unique()))
print("Months:", sorted(df_sales["Month"].unique()))
print("Weekdays:", df_sales["DayOfWeek"].nunique())

print("Missing YearMonth values:", df_sales["YearMonth"].isna().sum())

display(
    df_sales["YearMonth"]
    .value_counts()
    .sort_index()
    .head(10)
)

Years: [np.int32(2009), np.int32(2010), np.int32(2011)]
Months: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
Weekdays: 7
Missing YearMonth values: 0


YearMonth
2009-12    43453
2010-01    30317
2010-02    27952
2010-03    39841
2010-04    32858
2010-05    33383
2010-06    38391
2010-07    32112
2010-08    32109
2010-09    40630
Freq: M, Name: count, dtype: int64


## Step 3: Monthly Sales Analysis

Aggregate transaction data by month to calculate revenue, units sold, and unique invoices.


In [12]:
monthly_sales = (
    df_sales.groupby("YearMonth")
    .agg(
        Total_Revenue=("Revenue","sum"),
        Total_Units=("Quantity","sum"),
        Unique_Invoices=("Invoice","nunique"),
        Average_Line_Revenue=("Revenue","mean"),
    )
    .reset_index()
)

display(monthly_sales.head(10))

,YearMonth,Total_Revenue,Total_Units,Unique_Invoices,Average_Line_Revenue
0,2009-12,822483.950,425461,1682,18.928128
1,2010-01,651155.112,390672,1105,21.478217
2,2010-02,551504.726,381879,1201,19.730421
3,2010-03,830915.261,526026,1681,20.855783
4,2010-04,678875.252,366705,1462,20.660882
5,2010-05,657705.500,395868,1500,19.701809
6,2010-06,749537.310,406820,1645,19.523777
7,2010-07,648810.270,337895,1529,20.204605
8,2010-08,695251.910,472380,1425,21.652867
9,2010-09,921696.991,583864,1839,22.685134


In [11]:
# Month with the highest revenue
best_month = monthly_sales.loc[
    monthly_sales["Total_Revenue"].idxmax()
]

# Month with the lowest revenue
worst_month = monthly_sales.loc[
    monthly_sales["Total_Revenue"].idxmin()
]

print("Highest-revenue month:")
display(best_month)

print("Lowest-revenue month:")
display(worst_month)

Highest-revenue month:


YearMonth                  2011-11
Total_Revenue           1503866.78
Total_Units                 751377
Unique_Invoices               2769
Average_Line_Revenue     18.338944
Name: 23, dtype: object

Lowest-revenue month:


YearMonth                 2011-02
Total_Revenue           522545.56
Total_Units                282934
Unique_Invoices              1100
Average_Line_Revenue    19.438493
Name: 14, dtype: object